# AI-Powered Retail Business Analytics & Forecasting Platform

# Phase 4: Data Integration

## Objective

The Olist dataset consists of multiple relational tables containing information about customers, orders, products, payments, sellers, and reviews.

The objective of this notebook is to:

- Load cleaned datasets
- Understand relationships between tables
- Merge datasets using primary and foreign keys
- Create a master analytical dataset
- Validate the merged data
- Save the final integrated dataset

The final dataset will be used for all future analytics and machine learning tasks.

In [33]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [34]:
customers = pd.read_csv("C:/Users/rames/notebook/DS-1/data/cleaned/customers_cleaned.csv")

geolocation = pd.read_csv("C:/Users/rames/notebook/DS-1/data/cleaned/geolocation_cleaned.csv")

order_items = pd.read_csv("C:/Users/rames/notebook/DS-1/data/cleaned/order_items_cleaned.csv")

payments = pd.read_csv("C:/Users/rames/notebook/DS-1/data/cleaned/payments_cleaned.csv")

reviews = pd.read_csv("C:/Users/rames/notebook/DS-1/data/cleaned/reviews_cleaned.csv")

orders = pd.read_csv("C:/Users/rames/notebook/DS-1/data/cleaned/orders_cleaned.csv")

products = pd.read_csv("C:/Users/rames/notebook/DS-1/data/cleaned/products_cleaned.csv")

sellers = pd.read_csv("C:/Users/rames/notebook/DS-1/data/cleaned/sellers_cleaned.csv")

translation = pd.read_csv("C:/Users/rames/notebook/DS-1/data/cleaned/translation_cleaned.csv")

In [35]:
customers = pd.read_csv(
    "C:/Users/rames/notebook/DS-1/data/cleaned/customers_cleaned.csv"
)

In [36]:
orders = pd.read_csv(
    "C:/Users/rames/notebook/DS-1/data/cleaned/orders_cleaned.csv"
)

In [37]:
order_items = pd.read_csv(
    "C:/Users/rames/notebook/DS-1/data/cleaned/order_items_cleaned.csv"
)

In [38]:
products = pd.read_csv(
    "C:/Users/rames/notebook/DS-1/data/cleaned/products_cleaned.csv"
)

In [39]:
payments = pd.read_csv(
    "C:/Users/rames/notebook/DS-1/data/cleaned/payments_cleaned.csv"
)

In [40]:
reviews = pd.read_csv(
    "C:/Users/rames/notebook/DS-1/data/cleaned/reviews_cleaned.csv"
)

In [41]:
sellers = pd.read_csv(
    "C:/Users/rames/notebook/DS-1/data/cleaned/sellers_cleaned.csv"
)

In [42]:
translation = pd.read_csv(
    "C:/Users/rames/notebook/DS-1/data/cleaned/translation_cleaned.csv"
)

In [43]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Products": products,
    "Payments": payments,
    "Reviews": reviews,
    "Sellers": sellers,
    "Translation": translation
}


for name,df in datasets.items():

    print(name)

    print("Rows:",df.shape[0])

    print("Columns:",df.shape[1])

    print("-"*50)

Customers
Rows: 99441
Columns: 5
--------------------------------------------------
Orders
Rows: 99441
Columns: 8
--------------------------------------------------
Order Items
Rows: 112650
Columns: 7
--------------------------------------------------
Products
Rows: 32951
Columns: 9
--------------------------------------------------
Payments
Rows: 103886
Columns: 5
--------------------------------------------------
Reviews
Rows: 100000
Columns: 7
--------------------------------------------------
Sellers
Rows: 3095
Columns: 4
--------------------------------------------------
Translation
Rows: 71
Columns: 2
--------------------------------------------------


## Table Relationships


Customers
|
| customer_id
|
▼

Orders
|
| order_id
|
├──────────────► Payments
|
├──────────────► Reviews
|
▼

Order Items
|
| product_id
|
├──────────────► Products
|
|
| seller_id
|
▼

Sellers


Products
|
product_category_name
|
▼

Category Translation


In [84]:
translation.columns


Index(['product_category_name', 'product_category_name_english'], dtype='object')

In [85]:
master_df = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

print(master_df.shape)
print("Duplicate rows:", master_df.duplicated().sum())

(99441, 12)
Duplicate rows: 0


In [86]:
master_df = master_df.merge(
    order_items,
    on="order_id",
    how="left"
)
print(master_df.shape)
print("Duplicate rows:", master_df.duplicated().sum())

(113425, 18)
Duplicate rows: 0


In [87]:
master_df = master_df.merge(
    products,
    on="product_id",
    how="left"
)
print("After Products Merge")
print(master_df.shape)
print(master_df.isnull().sum().sum())

print(master_df.shape)
print("Duplicate rows:", master_df.duplicated().sum())

After Products Merge
(113425, 26)
10850
(113425, 26)
Duplicate rows: 0


In [88]:
master_df = master_df.merge(
    translation,
    on="product_category_name",
    how="left"
)

print(master_df.shape)
print("Duplicate rows:", master_df.duplicated().sum())


(113425, 27)
Duplicate rows: 0


In [89]:
master_df["product_category_name_english"] = (
    master_df["product_category_name_english"]
    .fillna("Unknown")
)

print(master_df.shape)
print("Duplicate rows:", master_df.duplicated().sum())

(113425, 27)
Duplicate rows: 0


In [90]:
master_df = master_df.merge(
    sellers,
    on="seller_id",
    how="left"
)

print(master_df.shape)
print("Duplicate rows:", master_df.duplicated().sum())

(113425, 30)
Duplicate rows: 0


In [91]:
#Payment Aggregation

payments_agg = payments.groupby(
    "order_id"
).agg({

    "payment_value":"sum",

    "payment_installments":"max",

    "payment_type":"first"

}).reset_index()

In [92]:
master_df = master_df.merge(
    payments_agg,
    on="order_id",
    how="left"
)
print("After payments agg Merge")
print(master_df.shape)
print(master_df.isnull().sum().sum())

print(master_df.shape)
print("Duplicate rows:", master_df.duplicated().sum())

After payments agg Merge
(113425, 33)
13184
(113425, 33)
Duplicate rows: 0


In [93]:
master_df["payment_type"] = (
    master_df["payment_type"]
    .fillna("Unknown")
)

master_df["payment_installments"] = (
    master_df["payment_installments"]
    .fillna(0)
)

master_df["payment_value"] = (
    master_df["payment_value"]
    .fillna(0)
)

In [94]:
reviews_selected = (
    reviews[
        ["order_id", "review_score"]
    ]
    .sort_values("order_id")
    .drop_duplicates(
        subset="order_id",
        keep="first"
    )
)

In [95]:
print("Duplicate order_ids in reviews_selected:")

print(
    reviews_selected
        .duplicated(subset="order_id")
        .sum()
)

Duplicate order_ids in reviews_selected:
0


In [96]:
master_df = master_df.merge(
    reviews_selected,
    on="order_id",
    how="left"
)
print("After review selected Merge")
print(master_df.shape)
print(master_df.isnull().sum().sum())

print(master_df.shape)
print("Duplicate rows:", master_df.duplicated().sum())

After review selected Merge
(113425, 34)
13175
(113425, 34)
Duplicate rows: 0


In [97]:
merge_validation = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Duplicate Rows",
        "Missing Values"
    ],
    "Value": [
        master_df.shape[0],
        master_df.shape[1],
        master_df.duplicated().sum(),
        master_df.isnull().sum().sum()
    ]
})

merge_validation

,Metric,Value
0,Rows,113425
1,Columns,34
2,Duplicate Rows,0
3,Missing Values,13175


In [98]:
master_df.shape

(113425, 34)

In [99]:
master_df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category_name_english', 'seller_zip_code_prefix',
       'seller_city', 'seller_state', 'payment_value', 'payment_installments',
       'payment_type', 'review_score'],
      dtype='object')

In [100]:
master_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_value,payment_installments,payment_type,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP,38.71,1.0,credit_card,4
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP,141.46,1.0,boleto,4
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP,179.12,3.0,credit_card,5
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG,72.20,1.0,credit_card,5
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP,28.62,1.0,credit_card,5


In [101]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 34 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       113425 non-null  object 
 1   customer_id                    113425 non-null  object 
 2   order_status                   113425 non-null  object 
 3   order_purchase_timestamp       113425 non-null  object 
 4   order_approved_at              113425 non-null  object 
 5   order_delivered_carrier_date   113425 non-null  object 
 6   order_delivered_customer_date  113425 non-null  object 
 7   order_estimated_delivery_date  113425 non-null  object 
 8   customer_unique_id             113425 non-null  object 
 9   customer_zip_code_prefix       113425 non-null  int64  
 10  customer_city                  113425 non-null  object 
 11  customer_state                 113425 non-null  object 
 12  order_item_id                 

In [102]:
missing_summary = (
    master_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[
    missing_summary > 0
]

missing_percent = (
    master_df.isnull().mean() * 100
).round(2)

missing_report = pd.DataFrame({
    "Missing Count": missing_summary,
    "Missing %": missing_percent[missing_summary.index]
})

missing_report

,Missing Count,Missing %
shipping_limit_date,775,0.68
seller_id,775,0.68
seller_state,775,0.68
product_height_cm,775,0.68
freight_value,775,0.68
price,775,0.68
order_item_id,775,0.68
product_id,775,0.68
seller_city,775,0.68
seller_zip_code_prefix,775,0.68


In [103]:
print(master_df.shape)
print("Duplicate rows:", master_df.duplicated().sum())

(113425, 34)
Duplicate rows: 0


In [104]:
date_columns = [

"order_purchase_timestamp",

"order_approved_at",

"order_delivered_carrier_date",

"order_delivered_customer_date",

"order_estimated_delivery_date"

]


for col in date_columns:

    master_df[col] = pd.to_datetime(
        master_df[col]
    )

In [105]:
master_df["delivery_days"] = (

master_df["order_delivered_customer_date"]

-

master_df["order_purchase_timestamp"]

).dt.days

In [106]:
master_df["delivery_delay_days"] = (
    master_df["order_delivered_customer_date"]
    -
    master_df["order_estimated_delivery_date"]
).dt.days

In [107]:
master_df["order_approved_at"] = (
    master_df["order_approved_at"]
    .fillna(master_df["order_purchase_timestamp"])
)

master_df["order_delivered_carrier_date"] = (
    master_df["order_delivered_carrier_date"]
    .fillna(master_df["order_estimated_delivery_date"])
)

master_df["order_delivered_customer_date"] = (
    master_df["order_delivered_customer_date"]
    .fillna(master_df["order_estimated_delivery_date"])
)

In [108]:
master_df["price"] = master_df["price"].fillna(0)
master_df["freight_value"] = master_df["freight_value"].fillna(0)

master_df["total_order_value"] = (
    master_df["price"] +
    master_df["freight_value"]
)

In [109]:
master_df["purchase_year"] = (

master_df["order_purchase_timestamp"]

.dt.year

)

In [110]:
master_df["purchase_month"] = (

master_df["order_purchase_timestamp"]

.dt.month

)

In [111]:
master_df.to_csv(
"C:/Users/rames/notebook/DS-1/data/processed/master_retail_dataset.csv",
index=False
)

In [112]:
print(
"Final Dataset Shape:",
master_df.shape
)

Final Dataset Shape: (113425, 39)


In [113]:
master_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_value,payment_installments,payment_type,review_score,delivery_days,delivery_delay_days,total_order_value,purchase_year,purchase_month
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP,38.71,1.0,credit_card,4,8,-8,38.71,2017,10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP,141.46,1.0,boleto,4,13,-6,141.46,2018,7
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP,179.12,3.0,credit_card,5,9,-18,179.12,2018,8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG,72.20,1.0,credit_card,5,13,-13,72.20,2017,11
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP,28.62,1.0,credit_card,5,2,-10,28.62,2018,2


# Conclusion

In this notebook, all cleaned Olist datasets were integrated into a single master analytical dataset.

Completed tasks:

- Loaded cleaned datasets
- Studied table relationships
- Performed relational joins
- Aggregated payment information
- Added customer, product, seller, and order information
- Created business-oriented features
- Saved final analytical dataset

The integrated dataset is now ready for Exploratory Data Analysis and Business Analytics.

In [114]:
print(master_df.duplicated().sum())

0


In [115]:
print("Customers :", customers["customer_id"].duplicated().sum())
print("Products  :", products["product_id"].duplicated().sum())
print("Sellers   :", sellers["seller_id"].duplicated().sum())
print("Payments  :", payments["order_id"].duplicated().sum())
print("Reviews   :", reviews["order_id"].duplicated().sum())
print("Translation :", translation["product_category_name"].duplicated().sum())

Customers : 0
Products  : 0
Sellers   : 0
Payments  : 4446
Reviews   : 559
Translation : 0
